# Main results (Tables 4–7)

## Setup

In [3]:
import sys, os
sys.path.insert(0, "../..")

from solar_ml_core import *

import numpy as np
import pandas as pd
from scipy import stats

CLIP = True
ALGOS = ["MLR", "KNN", "SVR", "RandomForest", "XGBoost", "LSTM"]
SITES = ["Atacama", "Bolzano", "Elazig", "Helsinki", "Islamabad", "Ningxia", "Yulara"]
CONTROL = "Reference"

## Metrics from saved predictions

In [4]:
def metrics(y_true, y_pred):
    err = y_pred - y_true
    ss_res = float(err @ err)
    ss_tot = float(((y_true - y_true.mean()) ** 2).sum())
    return {"R2": 1 - ss_res / ss_tot,
            "RMSE": float(np.sqrt(ss_res / len(y_true))),
            "MAE": float(np.abs(err).mean())}

rows = []
for a in ALGOS:
    for s in SITES:
        p = pd.read_csv(os.path.join(CFG.predictions_dir, f"{a}_{s}.csv"))
        if "used_in_metrics" in p.columns:
            p = p[p["used_in_metrics"]]
        y, yh = p["y_true"].to_numpy(float), p["y_pred"].to_numpy(float)
        if CLIP:
            yh = np.clip(yh, 0.0, None)
        rows.append({"Model": a, "Test_Station": s, "n": len(y), **metrics(y, yh)})

RES = pd.DataFrame(rows)

## Physical reference (k·POA)

In [5]:
folds = load_loso_folds(CFG)
rows = []
for site, (tr, te) in folds.items():
    tr = tr.dropna(subset=[CFG.poa_feature, CFG.target_col])
    te = te.dropna(subset=[CFG.poa_feature, CFG.target_col])
    x, y = tr[CFG.poa_feature].to_numpy(float), tr[CFG.target_col].to_numpy(float)
    k = (x @ y) / (x @ x)
    y_true = te[CFG.target_col].to_numpy(float)
    y_pred = np.clip(k * te[CFG.poa_feature].to_numpy(float), 0.0, None)
    rows.append({"Model": CONTROL, "Test_Station": site, "k": k,
                 "n": len(y_true), **metrics(y_true, y_pred)})

BASE = pd.DataFrame(rows).set_index("Test_Station").loc[SITES].reset_index()
display(BASE[["Test_Station", "k", "R2", "RMSE", "MAE"]].round(4))

RES7 = pd.concat([RES, BASE[RES.columns]], ignore_index=True)
ORDER = [CONTROL] + ALGOS

Loading LOSO folds
[Atacama   ] train 52,584 rows (6 stations) | test 8,676 rows | flagged %10.0
[Bolzano   ] train 52,500 rows (6 stations) | test 8,760 rows | flagged %10.1
[Elazig    ] train 52,500 rows (6 stations) | test 8,760 rows | flagged %10.1
[Helsinki  ] train 52,500 rows (6 stations) | test 8,760 rows | flagged %8.5
[Islamabad ] train 52,500 rows (6 stations) | test 8,760 rows | flagged %10.1
[Ningxia   ] train 52,500 rows (6 stations) | test 8,760 rows | flagged %10.0
[Yulara    ] train 52,476 rows (6 stations) | test 8,784 rows | flagged %10.1
7 folds.


,Test_Station,k,R2,RMSE,MAE
0,Atacama,0.0007,0.9340,0.0856,0.0482
1,Bolzano,0.0007,0.8842,0.1018,0.0547
2,Elazig,0.0008,0.7904,0.1337,0.0681
3,Helsinki,0.0008,0.9246,0.0553,0.0235
4,Islamabad,0.0008,0.9858,0.0265,0.0143
5,Ningxia,0.0008,0.9245,0.0630,0.0288
6,Yulara,0.0008,0.0807,0.1823,0.0866


## Table 4: overall comparison

In [6]:
T4 = (RES7.groupby("Model")[["R2", "RMSE", "MAE"]]
          .agg(["mean", "std", "median"]).loc[ORDER].round(4))
display(T4)

R2                    RMSE                     MAE          \
                mean     std  median    mean     std  median    mean     std   
Model                                                                          
Reference     0.7892  0.3182  0.9245  0.0926  0.0524  0.0856  0.0463  0.0259   
MLR           0.8307  0.1462  0.8608  0.0963  0.0394  0.1116  0.0637  0.0325   
KNN           0.7222  0.4484  0.8834  0.1023  0.0572  0.0760  0.0571  0.0329   
SVR           0.7304  0.2489  0.7632  0.1232  0.0800  0.0980  0.0887  0.0807   
RandomForest  0.7694  0.2293  0.8428  0.1092  0.0365  0.1077  0.0547  0.0182   
XGBoost       0.7939  0.2123  0.8447  0.1029  0.0409  0.1049  0.0520  0.0205   
LSTM          0.7020  0.1971  0.8177  0.1288  0.0415  0.1278  0.0748  0.0288   

                      
              median  
Model                 
Reference     0.0482  
MLR           0.0668  
KNN           0.0450  
SVR           0.0663  
RandomForest  0.0574  
XGBoost       0.0517  
LSTM          0.0721

## Tables 5 and 6: R² and skill score per station

In [7]:
T5 = RES7.pivot(index="Test_Station", columns="Model", values="R2").loc[SITES, ORDER]
display(T5.round(3))

ref_rmse = BASE.set_index("Test_Station")["RMSE"]
SS = (RES.pivot(index="Test_Station", columns="Model", values="RMSE")
         .loc[SITES, ALGOS].apply(lambda col: 1 - col / ref_rmse))
T6 = pd.concat([SS,
                SS.median().to_frame("Median").T,
                (SS > 0).sum().to_frame("Stations beating reference").T])
display(T6.round(3))

Model,Reference,MLR,KNN,SVR,RandomForest,XGBoost,LSTM
Test_Station,,,,,,,
Atacama,0.934,0.820,0.968,0.251,0.843,0.822,0.645
Bolzano,0.884,0.861,0.868,0.902,0.870,0.877,0.818
Elazig,0.790,0.828,0.793,0.756,0.793,0.806,0.833
Helsinki,0.925,0.885,0.896,0.763,0.863,0.845,0.422
Islamabad,0.986,0.966,0.883,0.959,0.930,0.934,0.869
Ningxia,0.925,0.933,0.934,0.911,0.828,0.946,0.876
Yulara,0.081,0.522,-0.287,0.571,0.258,0.328,0.451


Model,MLR,KNN,SVR,RandomForest,XGBoost,LSTM
Atacama,-0.649,0.306,-2.369,-0.543,-0.643,-1.319
Bolzano,-0.096,-0.068,0.079,-0.058,-0.031,-0.256
Elazig,0.094,0.006,-0.078,0.007,0.037,0.106
Helsinki,-0.234,-0.175,-0.772,-0.348,-0.436,-1.773
Islamabad,-0.547,-1.869,-0.698,-1.218,-1.163,-2.039
Ningxia,0.056,0.064,-0.088,-0.511,0.156,-0.286
Yulara,0.279,-0.183,0.317,0.102,0.145,0.228
Median,-0.096,-0.068,-0.088,-0.348,-0.031,-0.286
Stations beating reference,3.000,3.000,2.000,2.000,3.000,2.000


## Table 7: Friedman (Iman–Davenport) and Holm

In [8]:
def friedman_holm(metric, higher_is_better, alpha=0.05):
    M = RES7.pivot(index="Test_Station", columns="Model", values=metric).loc[SITES, ORDER]
    N, k = M.shape
    ranks = M.rank(axis=1, ascending=not higher_is_better).mean()
    chi2 = stats.friedmanchisquare(*[M[c] for c in M.columns]).statistic
    FF = (N - 1) * chi2 / (N * (k - 1) - chi2)
    p_ff = stats.f.sf(FF, k - 1, (k - 1) * (N - 1))
    print(f"{metric} | N={N}, k={k} | chi2={chi2:.2f} | F={FF:.2f}, p={p_ff:.4f}")
    print("mean ranks:", ranks.sort_values().round(2).to_dict())
    if p_ff >= alpha:
        return ranks, None
    SE = np.sqrt(k * (k + 1) / (6 * N))
    df = pd.DataFrame([{"Model": m, "Mean_rank": ranks[m], "Rank_diff": ranks[m] - ranks[CONTROL]}
                       for m in ORDER if m != CONTROL])
    df["z"] = df.Rank_diff / SE
    df["p"] = 2 * stats.norm.sf(df.z.abs())
    df = df.sort_values("p").reset_index(drop=True)
    df["Holm_threshold"] = [alpha / (k - 1 - i) for i in range(len(df))]
    sig, stop = [], False
    for p_, a_ in zip(df.p, df.Holm_threshold):
        s = (not stop) and p_ < a_
        stop = stop or not s
        sig.append(s)
    df["Significant"] = sig
    display(df.round(4))
    return ranks, df

ranks_r2, _ = friedman_holm("R2", True)
ranks_mae, holm_mae = friedman_holm("MAE", False)

R2 | N=7, k=7 | chi2=5.45 | F=0.89, p=0.5094
mean ranks: {'Reference': 3.14, 'MLR': 3.29, 'XGBoost': 3.43, 'KNN': 4.0, 'SVR': 4.29, 'RandomForest': 4.57, 'LSTM': 5.29}
MAE | N=7, k=7 | chi2=11.39 | F=2.23, p=0.0622
mean ranks: {'Reference': 2.57, 'XGBoost': 2.71, 'RandomForest': 3.71, 'MLR': 4.14, 'KNN': 4.14, 'SVR': 5.14, 'LSTM': 5.57}
